# Kaggle LLM server — Worker control plane + Open WebUI

Run a >8B model on Kaggle's free GPU, reach it through a **stable Cloudflare
Worker URL**, and Start/Stop it from a password-protected page that enables and
disables the GitHub workflow behind the scenes.

## Before you run (once, in Kaggle's right panel)
1. **Accelerator → GPU T4 x2** (or a single P100).
2. **Internet → ON**.
3. Add-ons → **Secrets**, add `WORKER_URL` and `WORKER_SECRET` (values from the
   Worker you deployed). Optional: `HF_TOKEN` if the model needs it.

Then **Run All**. The tunnel URL gets registered with the Worker automatically,
so Open WebUI can keep pointing at the same Worker URL forever.


## 1. Install (a few minutes, once per session)

In [ ]:
import subprocess, os, shutil

def sh(cmd, check=True):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=check)

sh("pip install -q fastapi 'uvicorn[standard]' httpx requests")

if not os.path.exists("/usr/local/bin/cloudflared"):
    sh("curl -fsSL -o /usr/local/bin/cloudflared "
       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    sh("chmod +x /usr/local/bin/cloudflared")

# zstd is required by the Ollama installer and is missing from the Kaggle image
if not shutil.which("zstd"):
    sh("apt-get update -qq && apt-get install -y -qq zstd", check=False)
    if not shutil.which("zstd"):
        sh("conda install -y -q -c conda-forge zstd", check=False)
print("zstd:", shutil.which("zstd") or "MISSING")

if not os.path.exists("/usr/local/bin/ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh")

print("\nready:", subprocess.run("cloudflared --version; ollama --version",
      shell=True, capture_output=True, text=True).stdout)


## 2. Config — edit these

`WORKER_URL`, `WORKER_SECRET`, `API_KEY` and `PANEL_PASS` are read from the
attached **`kaggle-llm-config` dataset** (`config.env`), falling back to Kaggle
Secrets then environment — so no credential sits in the notebook. **`API_KEY`
must be the exact same value as the Worker's `API_KEY`** — Open WebUI sends it
through the Worker to this session. The ntfy topic is random on purpose.


In [ ]:
# ================= CONFIG — edit =================
import os, secrets

def _from_input(name):
    # Read a KEY=value from config.env in an attached Kaggle dataset.
    import glob
    for p in glob.glob("/kaggle/input/**/config.env", recursive=True):
        try:
            for ln in open(p, encoding="utf-8", errors="ignore"):
                s = ln.strip()
                if s.startswith(name + "="):
                    return s.split("=", 1)[1].strip().strip('"').strip("'")
        except Exception:
            pass
    return None

def _cfg(name, default=""):
    # Config precedence: attached dataset -> Kaggle Secrets -> environment.
    v = _from_input(name)
    if v:
        return v
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, default)

MODEL          = "qwen2.5:14b"        # >8B. qwen2.5:32b needs T4 x2; also gemma2:9b, llama3.1:8b
PANEL_USER     = "bhumi"
PANEL_PASS     = _cfg("PANEL_PASS", "change-me-please")   # from Kaggle Secrets
API_KEY        = _cfg("API_KEY", "change-me-please")      # MUST equal the Worker's API_KEY
NTFY_TOPIC     = "kaggle-llm-" + secrets.token_hex(4)
NTFY_SERVER    = "https://ntfy.sh"
PANEL_PORT     = 8000
OLLAMA_PORT    = 11434
OLLAMA_MODELS  = "/kaggle/working/ollama-models"
WORKER_URL     = _cfg("WORKER_URL", "")       # https://<your-worker>.workers.dev
WORKER_SECRET  = _cfg("WORKER_SECRET", "")    # shared secret, same as the Worker
CONTROL_TOKEN  = secrets.token_hex(16)           # lets the Worker's Stop reach us
MAX_RUNTIME_H  = 9.5                             # self-terminate before Kaggle's 12 h cap
# ================================================

print("ntfy topic :", NTFY_TOPIC)
print("worker url :", WORKER_URL or "(not set — Worker registration will be skipped)")
print("config src : dataset / Kaggle Secrets / env")


## 3. Write the control panel

Serves the local Start/Stop page and proxies the OpenAI-compatible `/v1` that
Open WebUI (via the Worker) talks to. Panel = your password; `/v1` = bearer key;
`/admin/stop` = the Worker's control token.


In [ ]:
%%writefile control_panel.py
"""
control_panel.py — password-protected Start/Stop page + OpenAI-compatible
/v1 proxy, for a Kaggle-hosted Ollama model.

Everything is driven by environment variables (set in the notebook launch cell):
  PANEL_USER, PANEL_PASS, API_KEY, MODEL, OLLAMA_PORT, OLLAMA_MODELS,
  PANEL_PORT, NTFY_SERVER, NTFY_TOPIC, PUBLIC_URL
"""
import os
import base64
import threading
import subprocess
import time

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse
import uvicorn

CFG = dict(
    PANEL_USER=os.environ.get("PANEL_USER", "admin"),
    PANEL_PASS=os.environ.get("PANEL_PASS", "changeme"),
    API_KEY=os.environ.get("API_KEY", ""),
    MODEL=os.environ.get("MODEL", ""),
    OLLAMA_PORT=int(os.environ.get("OLLAMA_PORT", "11434")),
    OLLAMA_MODELS=os.environ.get("OLLAMA_MODELS", "/kaggle/working/ollama-models"),
    PANEL_PORT=int(os.environ.get("PANEL_PORT", "8000")),
    NTFY_SERVER=os.environ.get("NTFY_SERVER", "https://ntfy.sh"),
    NTFY_TOPIC=os.environ.get("NTFY_TOPIC", ""),
    PUBLIC_URL=os.environ.get("PUBLIC_URL", ""),
    CONTROL_TOKEN=os.environ.get("CONTROL_TOKEN", ""),
)

app = FastAPI()
state = {"status": "stopped", "detail": "idle — press Start"}
_lock = threading.Lock()
_ollama = None


# ---------------------------------------------------------------- notifications
def notify(msg, title="Kaggle LLM", tags="robot", priority="default"):
    if not CFG["NTFY_TOPIC"]:
        return
    try:
        httpx.post(
            f'{CFG["NTFY_SERVER"]}/{CFG["NTFY_TOPIC"]}',
            content=msg.encode(),
            headers={
                "Title": title,
                "Tags": tags,
                "Priority": priority,
                "Click": CFG["PUBLIC_URL"] or f'http://127.0.0.1:{CFG["PANEL_PORT"]}',
            },
            timeout=10,
        )
    except Exception as e:  # never let a notification failure break the server
        print("ntfy failed:", e)


# ------------------------------------------------------------------- auth
def _basic_ok(request: Request) -> bool:
    h = request.headers.get("authorization", "")
    if not h.lower().startswith("basic "):
        return False
    try:
        user, pw = base64.b64decode(h.split(" ", 1)[1]).decode().split(":", 1)
    except Exception:
        return False
    return user == CFG["PANEL_USER"] and pw == CFG["PANEL_PASS"]


def _bearer_ok(request: Request) -> bool:
    return bool(CFG["API_KEY"]) and request.headers.get("authorization", "") == f'Bearer {CFG["API_KEY"]}'


def _unauthorized():
    return JSONResponse(
        {"error": "unauthorized"},
        status_code=401,
        headers={"WWW-Authenticate": 'Basic realm="kaggle-llm"'},
    )


# ------------------------------------------------------------------- ollama
def _ollama_up() -> bool:
    try:
        return httpx.get(f'http://127.0.0.1:{CFG["OLLAMA_PORT"]}/api/tags', timeout=3).status_code == 200
    except Exception:
        return False


def _start():
    global _ollama
    with _lock:
        if state["status"] in ("running", "starting"):
            return
        state.update(status="starting", detail="launching ollama daemon")
    notify("Start requested — bringing the model up…", title="Kaggle LLM", tags="hourglass_flowing_sand", priority="high")

    env = os.environ.copy()
    env["OLLAMA_HOST"] = f'0.0.0.0:{CFG["OLLAMA_PORT"]}'
    env["OLLAMA_MODELS"] = CFG["OLLAMA_MODELS"]
    os.makedirs(CFG["OLLAMA_MODELS"], exist_ok=True)

    if not _ollama_up():
        log = open("/kaggle/working/ollama.log", "a")
        _ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
        for _ in range(60):
            if _ollama_up():
                break
            time.sleep(1)

    if not _ollama_up():
        state.update(status="error", detail="ollama did not come up — see /kaggle/working/ollama.log")
        notify("FAILED — ollama daemon did not start.", title="Kaggle LLM", tags="x", priority="urgent")
        return

    # Make sure the model is present (first Start pulls it; later Starts are instant).
    with _lock:
        state.update(status="starting", detail=f"ensuring {CFG['MODEL']} is downloaded")
    notify(f"Checking model {CFG['MODEL']} (first run downloads ~9 GB)…", title="Kaggle LLM", tags="inbox_tray", priority="default")
    subprocess.run(["ollama", "pull", CFG["MODEL"]], env=env, capture_output=True, text=True)

    # Warm it into VRAM with a 1-token generate.
    with _lock:
        state.update(status="starting", detail=f"loading {CFG['MODEL']} into GPU")
    try:
        httpx.post(
            f'http://127.0.0.1:{CFG["OLLAMA_PORT"]}/api/generate',
            json={"model": CFG["MODEL"], "prompt": "hi", "stream": False, "options": {"num_predict": 1}},
            timeout=600,
        )
    except Exception as e:
        print("warmup:", e)

    with _lock:
        state.update(status="running", detail=f"{CFG['MODEL']} is up")
    notify(
        f"{CFG['MODEL']} is UP.\nOpenAI base: {CFG['PUBLIC_URL']}/v1\nKey: {CFG['API_KEY']}",
        title="Kaggle LLM — running", tags="white_check_mark", priority="high",
    )


def _stop():
    global _ollama
    notify("Stop requested — shutting the model down…", title="Kaggle LLM", tags="octagonal_sign", priority="high")
    with _lock:
        state.update(status="stopping", detail="killing ollama")
    try:
        if _ollama and _ollama.poll() is None:
            _ollama.terminate()
            try:
                _ollama.wait(timeout=15)
            except Exception:
                _ollama.kill()
    except Exception as e:
        print("stop:", e)
    _ollama = None
    # belt and braces: kill any stray daemon
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    with _lock:
        state.update(status="stopped", detail="stopped — press Start to load again")
    notify("Model is DOWN. GPU freed.", title="Kaggle LLM — stopped", tags="zzz", priority="high")


# ------------------------------------------------------------------- routes
@app.get("/healthz")
def healthz():
    return {"ok": True, "status": state["status"]}


@app.get("/", response_class=HTMLResponse)
def index(request: Request):
    if not _basic_ok(request):
        return _unauthorized()
    color = {"running": "#0a7d32", "starting": "#b8860b", "stopping": "#b8860b",
             "stopped": "#666", "error": "#b00020"}.get(state["status"], "#333")
    html = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>Kaggle LLM control</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
 body{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:#0f1117;color:#e8eaed;margin:0;padding:2rem}}
 .card{{max-width:640px;margin:0 auto;background:#171a22;border:1px solid #262b36;border-radius:14px;padding:1.6rem}}
 h1{{font-size:1.25rem;margin:0 0 .3rem}} .sub{{color:#9aa0aa;font-size:.9rem;margin-bottom:1.2rem}}
 .status{{font-size:1rem;font-weight:600;color:{color};margin:.4rem 0 1rem}}
 .detail{{color:#9aa0aa;font-size:.85rem;margin-bottom:1.4rem}}
 button{{font:inherit;font-weight:600;border:0;border-radius:10px;padding:.8rem 1.4rem;cursor:pointer;margin-right:.6rem}}
 .start{{background:#0a7d32;color:#fff}} .stop{{background:#b00020;color:#fff}}
 button:disabled{{opacity:.4;cursor:not-allowed}}
 code{{background:#0b0d12;padding:.15rem .4rem;border-radius:6px;font-size:.85rem}}
 .kv{{margin-top:1.4rem;font-size:.85rem;line-height:1.9;color:#c7ccd4;word-break:break-all}}
</style></head>
<body><div class="card">
 <h1>Kaggle LLM server</h1>
 <div class="sub">model: <code>{CFG['MODEL']}</code></div>
 <div class="status">&#9679; {state['status'].upper()}</div>
 <div class="detail">{state['detail']}</div>
 <form method="post" action="/start" style="display:inline">
   <button class="start" {'disabled' if state['status'] in ('running','starting','stopping') else ''}>Start</button></form>
 <form method="post" action="/stop" style="display:inline">
   <button class="stop" {'disabled' if state['status'] in ('stopped','stopping') else ''}>Stop</button></form>
 <div class="kv">
   OpenAI base URL: <code>{CFG['PUBLIC_URL']}/v1</code><br>
   API key: <code>{CFG['API_KEY']}</code><br>
   ntfy topic: <code>{CFG['NTFY_TOPIC']}</code>
 </div>
</div>
<script>setTimeout(()=>location.reload(), 5000)</script>
</body></html>"""
    return HTMLResponse(html)


@app.post("/start")
def start(request: Request):
    if not _basic_ok(request):
        return _unauthorized()
    threading.Thread(target=_start, daemon=True).start()
    return HTMLResponse('<meta http-equiv="refresh" content="1;url=/">')


@app.post("/stop")
def stop(request: Request):
    if not _basic_ok(request):
        return _unauthorized()
    threading.Thread(target=_stop, daemon=True).start()
    return HTMLResponse('<meta http-equiv="refresh" content="1;url=/">')


@app.post("/admin/stop")
def admin_stop(request: Request):
    """Called by the Cloudflare Worker's Stop button (token auth, not Basic).
    Lets the control plane shut a live session down before it disables the
    GitHub workflow, so the model doesn't keep burning GPU until the timer."""
    if not CFG["CONTROL_TOKEN"] or request.headers.get("X-Control-Token") != CFG["CONTROL_TOKEN"]:
        return JSONResponse({"error": "forbidden"}, status_code=403)
    threading.Thread(target=_stop, daemon=True).start()
    return {"ok": True, "message": "stopping"}


@app.get("/status")
def status(request: Request):
    if not _basic_ok(request):
        return _unauthorized()
    return {"status": state["status"], "detail": state["detail"],
            "model": CFG["MODEL"], "openai_base": f'{CFG["PUBLIC_URL"]}/v1'}


# ------------------------------------------------- OpenAI-compatible proxy (/v1)
@app.api_route("/v1/{path:path}", methods=["GET", "POST", "OPTIONS"])
async def proxy(path: str, request: Request):
    if not _bearer_ok(request):
        return JSONResponse({"error": "invalid or missing API key"}, status_code=401)
    url = f'http://127.0.0.1:{CFG["OLLAMA_PORT"]}/v1/{path}'
    body = await request.body()
    headers = {"content-type": request.headers.get("content-type", "application/json")}
    client = httpx.AsyncClient(timeout=None)

    req = client.build_request(request.method, url, headers=headers, content=body)
    try:
        resp = await client.send(req, stream=True)
    except httpx.HTTPError:
        await client.aclose()
        return JSONResponse(
            {"error": {"message": f"Model is not running ({state['status']}). "
                                  f"Open the control page and press Start.", "type": "model_down"}},
            status_code=503,
        )

    async def gen():
        try:
            async for chunk in resp.aiter_raw():
                yield chunk
        finally:
            await resp.aclose()
            await client.aclose()

    return StreamingResponse(
        gen(),
        status_code=resp.status_code,
        media_type=resp.headers.get("content-type", "application/json"),
    )


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=CFG["PANEL_PORT"], log_level="warning")


## 4. Launch: tunnel → panel → register with the Worker → self-terminate timer

In [ ]:
import subprocess, re, time, os, threading, random

LOG = "/kaggle/working/cloudflared.log"
open(LOG, "w").close()
cf = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PANEL_PORT}"],
    stdout=open(LOG, "a"), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(LOG).read())
    if m:
        PUBLIC_URL = m.group(0)
        break

assert PUBLIC_URL, "cloudflared produced no URL — check /kaggle/working/cloudflared.log"
print("=" * 64)
print("  LOCAL PANEL :", PUBLIC_URL)
print("  OPENAI BASE :", PUBLIC_URL + "/v1")
print("=" * 64)

os.environ.update(
    PANEL_USER=PANEL_USER, PANEL_PASS=PANEL_PASS, API_KEY=API_KEY, MODEL=MODEL,
    OLLAMA_PORT=str(OLLAMA_PORT), OLLAMA_MODELS=OLLAMA_MODELS, PANEL_PORT=str(PANEL_PORT),
    NTFY_SERVER=NTFY_SERVER, NTFY_TOPIC=NTFY_TOPIC, PUBLIC_URL=PUBLIC_URL,
    CONTROL_TOKEN=CONTROL_TOKEN,
)

import control_panel, uvicorn
threading.Thread(
    target=lambda: uvicorn.run(control_panel.app, host="0.0.0.0", port=PANEL_PORT, log_level="warning"),
    daemon=True).start()
time.sleep(3)

# Register with the Cloudflare Worker so /v1 has a stable URL and the Worker's
# Stop button can reach this session.
if WORKER_URL:
    try:
        import requests
        r = requests.post(WORKER_URL.rstrip("/") + "/update-tunnel",
                          json={"url": PUBLIC_URL, "control_token": CONTROL_TOKEN},
                          headers={"X-Worker-Secret": WORKER_SECRET}, timeout=15)
        print("worker register:", r.status_code, r.text[:200])
    except Exception as e:
        print("worker register failed:", e)

# Bring the model up automatically — the session exists to serve it, so a
# registered tunnel should mean /v1 works without a second manual step.
threading.Thread(target=control_panel._start, daemon=True).start()

control_panel.notify(
    f"Kaggle box is up.\nTunnel: {PUBLIC_URL}\nPress START on the control page to load {MODEL}.",
    title="Kaggle LLM — session up", tags="white_check_mark", priority="high")

# Self-termination timer — stay safely under Kaggle's 12 h session cap.
def _self_terminate():
    secs = int(random.uniform(MAX_RUNTIME_H, MAX_RUNTIME_H + 0.5) * 3600)
    print(f"self-terminate scheduled in {secs/3600:.1f} h")
    time.sleep(secs)
    print("self-terminating now")
    try:
        control_panel._stop()
    except Exception:
        pass
    os._exit(0)

threading.Thread(target=_self_terminate, daemon=True).start()
print("\nNow open the Worker control page and press Start.\n")

while True:
    time.sleep(300)
    print("alive", time.strftime("%H:%M:%S"), "| model:", control_panel.state["status"])


## 5. Point Open WebUI at the Worker

In Open WebUI: **Settings → Admin → Connections → ➕ Add Connection → Standard/Compatible**

| Field | Value |
|---|---|
| API URL | `https://<your-worker>.workers.dev/v1` |
| API Key | your `API_KEY` (`sk-bhumi-local`) |

This never changes — the Worker follows whatever tunnel the current session
registered, so you don't touch Open WebUI again when the session restarts.

**Start / Stop** happens on the Worker's control page (`https://<your-worker>.workers.dev/`):
- **Start** enables the GitHub workflow and dispatches it (with an optional random delay).
- **Stop** asks this session to shut down, then disables the workflow so the cron can't restart it.

**ntfy:** subscribe in the app to server `ntfy.sh`, topic = the printed value.

**Caching the model:** models under `/kaggle/working` are lost when the session
ends. Save `ollama-models` as a Kaggle Dataset and attach it as input, then point
`OLLAMA_MODELS` at the dataset path — otherwise every session re-downloads ~9 GB.
